# 05 - Flood & Drought Risk Mapping

This notebook demonstrates:
- Computing flood and drought risk scores per catchment
- Generating static risk maps
- Creating interactive Folium maps
- Exporting GeoJSON for GIS integration
- Basin-level risk summaries

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import CAMELSIndDataLoader
from src.data.preprocessor import StreamflowPreprocessor
from src.features.engineer import FeatureEngineer
from src.visualization.risk_maps import RiskMapGenerator
from src.utils.helpers import set_seed

set_seed(42)
%matplotlib inline

In [ ]:
# Prepare data and model
loader = CAMELSIndDataLoader('../data/raw')
if not loader.list_catchments():
    loader.generate_sample_data(n_catchments=10, n_days=2000)

data = loader.load_all_catchments()
engineer = FeatureEngineer()
data = engineer.transform(data)

preprocessor = StreamflowPreprocessor()
X_train, X_test, y_train, y_test = preprocessor.fit_transform(data)

from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(f'Model trained. Test samples: {len(y_test)}')

In [ ]:
# Compute risk scores
attrs = pd.read_csv('../data/raw/catchment_attributes.csv', index_col=0)
attrs.index = attrs.index.astype(str)

test_data = data.iloc[-len(y_test):]
catchment_ids = test_data['catchment_id'].values if 'catchment_id' in test_data.columns else ['unknown'] * len(y_test)

risk_mapper = RiskMapGenerator({'output_dir': '../outputs/risk_maps'})
risk_df = risk_mapper.compute_risk_scores(model, X_test, catchment_ids, attrs)

print(f'Risk scores for {len(risk_df)} catchments:')
risk_df[['catchment_id', 'flood_risk_level', 'drought_risk_level', 'flood_risk_score', 'drought_risk_score']]

In [ ]:
# Generate risk maps
risk_mapper.generate_static_map(risk_df, risk_type='flood')
risk_mapper.generate_static_map(risk_df, risk_type='drought')
risk_mapper.generate_interactive_map(risk_df)
risk_mapper.export_geojson(risk_df)

summary = risk_mapper.generate_summary_report(risk_df)
print('\nRisk Distribution:')
print(f'  Flood: {summary["flood_risk_distribution"]}')
print(f'  Drought: {summary["drought_risk_distribution"]}')